# 1.4 — 3-Tier Aggressive Augmentation Strategy
Applies augmentation to the Training split ONLY.
- Tier 1: Keras-style geometric transforms (rotation, zoom, shift, shear)
- Tier 2: ElasticDistortion class (cv2.remap bicubic) - simulates motor-control micro-tremors
- Tier 3: Photometric perturbations (Gaussian noise/blur, brightness jitter, salt-and-pepper)
Saves augmented images to data/augmented/ with source-tagged filenames.
Spot-check grid saved to reports/spotcheck/.
All hyperparameters in config.py for full reproducibility.

In [5]:
import cv2, csv, random, pathlib, sys
import numpy as np

sys.path.insert(0, str(pathlib.Path('.').resolve()))
import config

random.seed(config.SEED)
np.random.seed(config.SEED)

MANIFEST_DIR  = pathlib.Path('../' + config.MANIFESTS_DIR)
PROCESSED_DIR = pathlib.Path('../' + config.PROCESSED_DIR)
AUG_DIR       = pathlib.Path('../' + config.AUGMENTED_DIR)
SPOTCHECK_DIR = pathlib.Path('../' + config.SPOTCHECK_DIR)
AUG_DIR.mkdir(parents=True, exist_ok=True)
SPOTCHECK_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_CSV = MANIFEST_DIR / 'train.csv'
print(f'TRAIN manifest : {TRAIN_CSV}')
print(f'Augmented output: {AUG_DIR}')

TRAIN manifest : ..\data\manifests\train.csv
Augmented output: ..\data\augmented


In [6]:
train_samples = []
with open(TRAIN_CSV, newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        train_samples.append((row['image_path'], int(row['label'])))
print(f'Training set size: {len(train_samples)} images')

Training set size: 124 images


In [7]:
# Tier 1: Geometric transforms (mirrors Keras ImageDataGenerator params from config)
def apply_tier1(img, rng):
    p = config.TIER1_PARAMS
    h, w = img.shape[:2]
    cx, cy = w / 2, h / 2
    angle = rng.uniform(-p['rotation_range'], p['rotation_range'])
    zoom  = rng.uniform(1 - p['zoom_range'], 1 + p['zoom_range'])
    M = cv2.getRotationMatrix2D((cx, cy), angle, zoom)
    M[0, 2] += rng.uniform(-p['width_shift_range'],  p['width_shift_range'])  * w
    M[1, 2] += rng.uniform(-p['height_shift_range'], p['height_shift_range']) * h
    out = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_REPLICATE)
    shear = rng.uniform(-p['shear_range'], p['shear_range'])
    S = np.float32([[1, shear, 0], [0, 1, 0]])
    out = cv2.warpAffine(out, S, (w, h), flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_REPLICATE)
    # horizontal_flip=False -- letter orientation must be preserved
    return out

print('Tier 1 (geometric) defined. horizontal_flip=False: letter orientation preserved.')

Tier 1 (geometric) defined. horizontal_flip=False: letter orientation preserved.


In [8]:
# Tier 2: ElasticDistortion class
class ElasticDistortion:
    '''Simulates motor-control micro-tremors via elastic deformation.

    Control-point grid at 40-80px intervals, random bounded displacement vectors,
    bicubic interpolation via cv2.remap with INTER_CUBIC.
    Lock alpha/sigma values in config.py after visual spot-check calibration.
    Use lower alpha for training to avoid distorting clinical biomarker features.
    '''
    def __init__(self, alpha=None, sigma=None):
        p = config.ELASTIC_PARAMS
        self.alpha = alpha if alpha is not None else p['alpha']
        self.sigma = sigma if sigma is not None else p['sigma']

    def __call__(self, img, rng):
        h, w = img.shape[:2]
        dx = (rng.random((h, w)) * 2 - 1) * self.alpha
        dy = (rng.random((h, w)) * 2 - 1) * self.alpha
        ksize = int(6 * self.sigma + 1) | 1
        dx = cv2.GaussianBlur(dx.astype(np.float32), (ksize, ksize), self.sigma)
        dy = cv2.GaussianBlur(dy.astype(np.float32), (ksize, ksize), self.sigma)
        x, y = np.meshgrid(np.arange(w), np.arange(h))
        map_x = (x + dx).astype(np.float32)
        map_y = (y + dy).astype(np.float32)
        return cv2.remap(img, map_x, map_y,
                         interpolation=cv2.INTER_CUBIC,
                         borderMode=cv2.BORDER_REPLICATE)

print('ElasticDistortion class defined.')

ElasticDistortion class defined.


In [9]:
# Tier 3: Photometric perturbations
def apply_tier3(img, rng):
    '''Gaussian noise (mu=0, sigma^2=0.01), Gaussian blur (3x3),
    brightness jitter +/-15%, salt-and-pepper noise (prob=0.005).'''
    p = config.TIER3_PARAMS
    out = img.astype(np.float32)
    noise = rng.normal(p['gaussian_noise_mean'], p['gaussian_noise_sigma'],
                       out.shape).astype(np.float32)
    out = np.clip(out + noise * 255, 0, 255)
    out = cv2.GaussianBlur(out, p['gaussian_blur_kernel'], 0)
    factor = rng.uniform(1 - p['brightness_jitter'], 1 + p['brightness_jitter'])
    out = np.clip(out * factor, 0, 255)
    sp = p['salt_pepper_prob']
    out[rng.random(out.shape) < sp / 2] = 255
    out[rng.random(out.shape) < sp / 2] = 0
    return out.astype(np.uint8)

print('Tier 3 (photometric) defined.')

Tier 3 (photometric) defined.


In [10]:
# AugmentationPipeline: combines all tiers; raises error if called on val/test
class AugmentationPipeline:
    '''Combines Tier 1+2+3 augmentation.
    Raises AssertionError if called on val or test data.
    All hyperparameters stored in config.py for full reproducibility.
    '''
    def __init__(self, split='train'):
        assert split == 'train', (
            f'AugmentationPipeline must ONLY be applied to the training split. '
            f'Got split={split!r}. Augmentation on val/test data is forbidden.'
        )
        self.split   = split
        self.elastic = ElasticDistortion(alpha=5.0)  # conservative: lower alpha

    def __call__(self, img, rng):
        out = apply_tier1(img, rng)
        out = self.elastic(out, rng)
        out = apply_tier3(out, rng)
        return out

pipeline = AugmentationPipeline(split='train')
print('AugmentationPipeline ready.')

# Assert guard works on val/test
try:
    AugmentationPipeline(split='val')
    raise RuntimeError('Guard failed!')
except AssertionError as e:
    print(f'Guard confirmed: {e}')

AugmentationPipeline ready.
Guard confirmed: AugmentationPipeline must ONLY be applied to the training split. Got split='val'. Augmentation on val/test data is forbidden.


In [11]:
# Generate augmented pool: 10 variants per training image (>= 10x)
rng = np.random.default_rng(config.SEED)
AUGMENTED_PER_IMAGE = config.MIN_AUG_MULTIPLIER

aug_manifest_rows = []
spotcheck_paths   = []
total_generated   = 0

print(f'Generating {AUGMENTED_PER_IMAGE} augmented variants per image ...')

for img_path_str, label in train_samples:
    img_path = pathlib.Path(img_path_str)
    bgr = cv2.imread(str(img_path))
    if bgr is None:
        print(f'  WARNING: Cannot read {img_path}')
        continue
    class_name = img_path.parent.name
    out_dir = AUG_DIR / class_name
    out_dir.mkdir(parents=True, exist_ok=True)
    for i in range(AUGMENTED_PER_IMAGE):
        aug      = pipeline(bgr, rng)
        out_name = f'{img_path.stem}_aug{i:03d}.jpg'  # source-tagged filename
        out_path = out_dir / out_name
        cv2.imwrite(str(out_path), aug)
        aug_manifest_rows.append((str(out_path.resolve()), label, True))
        total_generated += 1
        if len(spotcheck_paths) < 30 and i < 3:
            spotcheck_paths.append((str(out_path), label))

print(f'Total augmented: {total_generated}')
print(f'Multiplier     : {total_generated / max(len(train_samples),1):.1f}x original')
assert total_generated >= 20_000, \
    f'Pool too small! Got {total_generated}, need at least 20,000.'
print(f'Augmented pool meets >=20k requirement ({total_generated} images, '
      f'{total_generated / len(train_samples):.1f}x original).')

Generating 162 augmented variants per image ...


KeyboardInterrupt: 

In [8]:
# Write augmented training manifest
aug_train_csv = MANIFEST_DIR / 'train_augmented.csv'
with open(aug_train_csv, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['image_path', 'label', 'augmented'])
    for path, label in train_samples:
        writer.writerow([path, label, False])
print(f'Rows: {len(aug_manifest_rows)} augmented images (target: ≥20,000)')

Rows: 20088 augmented images (target: ≥20,000)


In [9]:
# Spot-check grid: 5x6 = 30 augmented samples; save to reports/spotcheck/
# Manually verify: stroke edges, spacing, slant, ink density are not destroyed
GRID_ROWS, GRID_COLS, THUMB = 5, 6, 150
grid = np.zeros((GRID_ROWS * THUMB, GRID_COLS * THUMB, 3), dtype=np.uint8)
label_color = {0: (0, 255, 0), 1: (0, 0, 255)}  # green=LPD, red=PD

for idx, (path, label) in enumerate(spotcheck_paths[:GRID_ROWS * GRID_COLS]):
    img = cv2.imread(path)
    if img is None: continue
    thumb = cv2.resize(img, (THUMB, THUMB))
    r, c = idx // GRID_COLS, idx % GRID_COLS
    grid[r*THUMB:(r+1)*THUMB, c*THUMB:(c+1)*THUMB] = thumb
    cv2.rectangle(grid, (c*THUMB, r*THUMB), ((c+1)*THUMB-1, (r+1)*THUMB-1),
                  label_color.get(label, (255,255,255)), 2)

spotcheck_out = SPOTCHECK_DIR / 'augmented_spotcheck.jpg'
cv2.imwrite(str(spotcheck_out), grid)
print(f'Spot-check grid saved: {spotcheck_out}')
print('Green border = LPD (0), Red border = PD (1)')
print('Manual verification required: check stroke edges, spacing, slant, ink density.')

Spot-check grid saved: ..\reports\spotcheck\augmented_spotcheck.jpg
Green border = LPD (0), Red border = PD (1)
Manual verification required: check stroke edges, spacing, slant, ink density.


In [10]:
# ── Save sample validation images per augmentation tier ───────────────────────
VALIDATION_DIR = pathlib.Path('../reports/validation')
AUG_STEPS = ['step6_tier1_geometric', 'step7_tier2_elastic',
             'step8_tier3_photometric', 'step9_all_tiers_combined']
for s in AUG_STEPS:
    (VALIDATION_DIR / s).mkdir(parents=True, exist_ok=True)

sample_paths = [
    pathlib.Path('../data/raw/mendeley/DATASET DYSGRAPHIA HANDWRITING/Low Potential Dysgraphia/LPD (1).jpg'),
    pathlib.Path('../data/raw/mendeley/DATASET DYSGRAPHIA HANDWRITING/Low Potential Dysgraphia/LPD (2).jpg'),
    pathlib.Path('../data/raw/mendeley/DATASET DYSGRAPHIA HANDWRITING/Potential Dysgraphia/PD (1).jpg'),
]

val_rng = np.random.default_rng(42)
elastic_val = ElasticDistortion(alpha=5.0)  # ← defined here directly

for img_path in sample_paths:
    name = img_path.stem
    bgr  = cv2.imread(str(img_path))

    # Preprocess first (same as 1.3 pipeline)
    gray     = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    _, bin_  = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, inv   = cv2.threshold(bin_, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    base     = cv2.resize(inv, (224, 224), interpolation=cv2.INTER_LINEAR)

    # Tier 1 only
    t1 = apply_tier1(base, val_rng)
    cv2.imwrite(str(VALIDATION_DIR / 'step6_tier1_geometric'    / f'{name}.jpg'), t1)

    # Tier 2 only
    elastic_val = ElasticDistortion(alpha=5.0)
    t2 = elastic_val(base, val_rng)
    cv2.imwrite(str(VALIDATION_DIR / 'step7_tier2_elastic'      / f'{name}.jpg'), t2)

    # Tier 3 only
    t3 = apply_tier3(base, val_rng)
    cv2.imwrite(str(VALIDATION_DIR / 'step8_tier3_photometric'  / f'{name}.jpg'), t3)

    # All combined
    combined = apply_tier1(base, val_rng)
    combined = elastic_val(combined, val_rng)
    combined = apply_tier3(combined, val_rng)
    cv2.imwrite(str(VALIDATION_DIR / 'step9_all_tiers_combined' / f'{name}.jpg'), combined)

    print(f'  Saved augmentation validation for: {name}')

print(f'\nAugmentation validation images saved to: {VALIDATION_DIR}')

  Saved augmentation validation for: LPD (1)
  Saved augmentation validation for: LPD (2)
  Saved augmentation validation for: PD (1)

Augmentation validation images saved to: ..\reports\validation


In [14]:
# ── Validation & Spotcheck: original-size samples per augmentation tier ───────
import cv2, numpy as np, pathlib

REPORTS_DIR   = pathlib.Path('../reports')
VALIDATION_DIR = REPORTS_DIR / 'validation'
SPOTCHECK_DIR  = REPORTS_DIR / 'spotcheck'

# Sub-folders: one per tier + combined
AUG_TIERS = [
    'tier1_geometric',
    'tier2_elastic',
    'tier3_photometric',
    'all_tiers_combined',
]
for t in AUG_TIERS:
    (VALIDATION_DIR / t).mkdir(parents=True, exist_ok=True)
SPOTCHECK_DIR.mkdir(parents=True, exist_ok=True)

# ── 3 sample images: 2 LPD, 1 PD ─────────────────────────────────────────────
RAW_DIR = pathlib.Path('../data/raw/mendeley/DATASET DYSGRAPHIA HANDWRITING')
sample_paths = [
    (RAW_DIR / 'Low Potential Dysgraphia' / 'LPD (1).jpg', 0),
    (RAW_DIR / 'Low Potential Dysgraphia' / 'LPD (2).jpg', 0),
    (RAW_DIR / 'Potential Dysgraphia'     / 'PD (1).jpg',  1),
]

val_rng     = np.random.default_rng(42)
elastic_val = ElasticDistortion(alpha=5.0)

for img_path, label in sample_paths:
    name = img_path.stem
    bgr  = cv2.imread(str(img_path))
    if bgr is None:
        print(f'  WARNING: Cannot read {img_path}')
        continue

    orig_h, orig_w = bgr.shape[:2]   # preserve original dimensions

    # Preprocess to binary (same pipeline as 1.3) but keep original size
    gray    = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    _, bin_ = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # Resize to original dims (not 224x224) for validation display
    base    = cv2.resize(bin_, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)
    base_bgr = cv2.cvtColor(base, cv2.COLOR_GRAY2BGR)

    # Tier 1 — geometric transforms only
    t1      = apply_tier1(base, val_rng)
    t1_bgr  = cv2.cvtColor(t1, cv2.COLOR_GRAY2BGR) if t1.ndim == 2 else t1
    cv2.imwrite(str(VALIDATION_DIR / 'tier1_geometric'     / f'{name}.jpg'), t1_bgr)

    # Tier 2 — elastic distortion only
    t2      = elastic_val(base, val_rng)
    t2_bgr  = cv2.cvtColor(t2, cv2.COLOR_GRAY2BGR) if t2.ndim == 2 else t2
    cv2.imwrite(str(VALIDATION_DIR / 'tier2_elastic'       / f'{name}.jpg'), t2_bgr)

    # Tier 3 — photometric only
    t3      = apply_tier3(base, val_rng)
    t3_bgr  = cv2.cvtColor(t3, cv2.COLOR_GRAY2BGR) if t3.ndim == 2 else t3
    cv2.imwrite(str(VALIDATION_DIR / 'tier3_photometric'   / f'{name}.jpg'), t3_bgr)

    # All tiers combined
    combined = apply_tier1(base, val_rng)
    combined = elastic_val(combined, val_rng)
    combined = apply_tier3(combined, val_rng)
    comb_bgr = cv2.cvtColor(combined, cv2.COLOR_GRAY2BGR) if combined.ndim == 2 else combined
    cv2.imwrite(str(VALIDATION_DIR / 'all_tiers_combined'  / f'{name}.jpg'), comb_bgr)

    print(f'  Saved validation samples for: {name}  '
          f'(original size: {orig_w}x{orig_h}px)')

print(f'\nValidation images saved to: {VALIDATION_DIR}')

# ── Spotcheck grid: 5x6 = 30 augmented samples ────────────────────────────────
# Samples are drawn from aug_manifest_rows generated in the previous cell
GRID_ROWS, GRID_COLS, THUMB = 5, 6, 150
grid = np.zeros((GRID_ROWS * THUMB, GRID_COLS * THUMB, 3), dtype=np.uint8)
label_color = {0: (0, 255, 0), 1: (0, 0, 255)}   # green=LPD, red=PD

spotcheck_sample = aug_manifest_rows[:GRID_ROWS * GRID_COLS]
for idx, (path, label, _) in enumerate(spotcheck_sample):
    img = cv2.imread(path)
    if img is None:
        continue
    thumb = cv2.resize(img, (THUMB, THUMB))
    r, c  = idx // GRID_COLS, idx % GRID_COLS
    grid[r*THUMB:(r+1)*THUMB, c*THUMB:(c+1)*THUMB] = thumb
    cv2.rectangle(grid,
                  (c*THUMB,      r*THUMB),
                  ((c+1)*THUMB-1, (r+1)*THUMB-1),
                  label_color.get(label, (255, 255, 255)), 3)

spotcheck_out = SPOTCHECK_DIR / 'augmented_spotcheck.jpg'
cv2.imwrite(str(spotcheck_out), grid)
print(f'Spot-check grid saved: {spotcheck_out}')
print('Green border = LPD (0)  |  Red border = PD (1)')
print('Manual check: stroke edges, spacing, slant, ink density.')

  Saved validation samples for: LPD (1)  (original size: 506x55px)
  Saved validation samples for: LPD (2)  (original size: 700x104px)
  Saved validation samples for: PD (1)  (original size: 1318x146px)

Validation images saved to: ..\reports\validation
Spot-check grid saved: ..\reports\spotcheck\augmented_spotcheck.jpg
Green border = LPD (0)  |  Red border = PD (1)
Manual check: stroke edges, spacing, slant, ink density.
